# 17장 깊게 쌓기 — VGG와 ResNet

**3단계 · 다시 만들기** — 『법률가를 위한 인공지능 코딩 자습서』 실습 노트북

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch17_3_build.ipynb)

이 노트북에는 **주석만 남긴 코드**가 들어 있습니다. 책과 1단계 노트북을 덮고, 주석을 실마리 삼아 코드를 되살려 보세요. 아래 표의 「다시 만들기」 과제가 이 노트북의 목표입니다.

- 코드 셀 첫 줄의 `%%writefile 파일이름` 은 "이 셀의 내용을 그 파일로 저장하라"는 뜻입니다. 채운 뒤 셀을 실행해 저장하고, 확인 셀로 돌려 봅니다.
- 막히면 오류 메시지를 먼저 읽고(부록 A), 그래도 안 되면 1단계 노트북에서 그 부분만 확인하세요.

| 단계 | 할 일 | 노트북 |
|---|---|---|
| 1. 따라 하기 | `vgg.py`, `resnet.py`, `p01_depth_experiment.py` 를 치고 실행 | [ch17_1_follow.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch17_1_follow.ipynb) |
| 2. 주석 달기 | `BasicBlock.forward` 세 줄과 `shortcut` 조건문에 "모양이 어떻게 되는가"를 주석으로 | [ch17_2_comment.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch17_2_comment.ipynb) |
| 3. 다시 만들기 | 주석만 남긴 파일에서 `BasicBlock` 복원. 그다음 14장 패키지 `networks/` 에 `vgg.py`·`resnet.py` 를 넣고 `--model resnet` 으로 학습되게 등록 | [ch17_3_build.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch17_3_build.ipynb) **(지금 이 노트북)** |

In [ ]:
# 준비 셀 — 맨 먼저 한 번 실행하세요. 실습 코드와 데이터를 받아 옵니다.
import os, sys
if "google.colab" in sys.modules:                       # Colab 에서 열었을 때
    ROOT = "/content/AI-Coding-for-Lawyers"
    if not os.path.isdir(ROOT):
        !git clone -q --depth 1 https://github.com/BowonKwon/AI-Coding-for-Lawyers.git {ROOT}
else:                                                   # 내 컴퓨터에서 열었을 때 (저장소 폴더 안)
    ROOT = os.getcwd()
    while not os.path.isdir(os.path.join(ROOT, "notebooks")) and os.path.dirname(ROOT) != ROOT:
        ROOT = os.path.dirname(ROOT)
os.environ.setdefault("JUCOS_DATA", os.path.join(ROOT, "data"))   # 데이터를 두는 폴더
os.chdir(os.path.join(ROOT, "code", "17"))                      # 이 장의 코드 폴더로 이동
if not os.path.isdir(os.path.join(os.environ["JUCOS_DATA"], "shapes")):   # 15장의 도형 데이터가 없으면 만든다
    !python ../15/make_shapes.py
print("준비 끝. 작업 폴더: code/17")

## `resnet.py` — 대상: `BasicBlock`

In [ ]:
%%writefile resnet.py
# 17.2 ResNet: 블록의 입력을 출력에 더하는 "지름길(skip connection)" 로 아주 깊게 쌓아도 학습되게 한다
import torch
import torch.nn as nn
import torch.nn.functional as F


class BasicBlock(nn.Module):
    """3x3 합성곱 둘 + 지름길. out = relu( F(x) + x )"""

    def __init__(self, in_ch, out_ch, stride=1):
        # 지름길: 모양(채널·크기)이 바뀌는 블록에서는 1x1 합성곱으로 x 의 모양을 맞춘다
        raise NotImplementedError("여기를 채우세요")

    def forward(self, x):
        # 더하기 한 줄이 ResNet 의 전부
        raise NotImplementedError("여기를 채우세요")


class ResNet(nn.Module):
    """입력부 → 단계(stage) 몇 개 → 출력부. 단계마다 채널은 두 배, 크기는 절반."""

    def __init__(self, blocks_per_stage=(2, 2, 2), channels=(16, 32, 64), in_ch=3, num_classes=10):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(in_ch, channels[0], 3, padding=1, bias=False),
                                  nn.BatchNorm2d(channels[0]), nn.ReLU(inplace=True))
        stages, ch = [], channels[0]
        for i, (n, out_ch) in enumerate(zip(blocks_per_stage, channels)):
            stride = 1 if i == 0 else 2                       # 첫 단계 외에는 크기 절반
            layers = [BasicBlock(ch, out_ch, stride)] + [BasicBlock(out_ch, out_ch) for _ in range(n - 1)]
            stages.append(nn.Sequential(*layers))
            ch = out_ch
        self.stages = nn.Sequential(*stages)
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(1), nn.Linear(ch, num_classes))

    def forward(self, x):
        return self.head(self.stages(self.stem(x)))


if __name__ == "__main__":
    for cfg in [(1, 1, 1), (2, 2, 2), (3, 3, 3)]:
        m = ResNet(cfg)
        n = sum(p.numel() for p in m.parameters())
        depth = 1 + 2 * sum(cfg) + 1
        print(f"ResNet 블록 {cfg}  합성곱 층 수 약 {depth:>2}  파라미터 {n:>9,}  [2,3,32,32] → {tuple(m(torch.zeros(2, 3, 32, 32)).shape)}")
    # 지름길이 모양을 맞추는지
    b = BasicBlock(16, 32, stride=2)
    print("BasicBlock(16→32, stride 2): [2,16,32,32] →", tuple(b(torch.zeros(2, 16, 32, 32)).shape))

### 확인

In [ ]:
!python resnet.py

### 원래대로 되돌리기

파일을 책의 원래 코드로 되돌리고 싶으면 아래 셀을 실행하세요. (Colab 에서 받은 저장소 기준입니다.)

In [ ]:
!git checkout -- resnet.py

## 그다음

14장 패키지 `networks/` 에 `vgg.py`·`resnet.py` 를 넣고 `--model resnet` 으로 학습되게 등록하세요. 이 과제는 14장 패키지(`code/14`)를 고치는 일입니다. 왼쪽 폴더 아이콘에서 `code/14` 의 파일을 열어 고치거나, 셀 첫 줄에 `%%writefile ../14/networks/새파일.py` 처럼 적어 저장한 뒤 아래 셀로 확인하세요.

In [ ]:
# 여기에 이어서 작성하세요